In [16]:
import os
from pathlib import Path

cwd = Path.cwd()

if cwd.name == "notebooks":
    os.chdir(cwd.parent)

In [17]:
import pandas as pd
from torch_geometric.loader import DataLoader
from gjepa.datasets.cv_vis.tmqmg_star import TMQMGStarDataset
from ase import Atoms
import torch
from torch.utils.data import Subset
from fairchem.core import pretrained_mlip, FAIRChemCalculator  

In [ ]:
def pyg_to_ase(data):
    pos = data.pos.cpu().numpy()
    numbers = data.z.cpu().numpy()
    return Atoms(numbers=numbers, positions=pos)


def run_inference_for_task(calc, dataloader):
    records = []

    for data in dataloader:
        ase_atoms = pyg_to_ase(data)
        ase_atoms.calc = calc

        # Always available
        E = ase_atoms.get_potential_energy()
        F = ase_atoms.get_forces()

        # Stress may not be implemented for some tasks
        try:
            S = ase_atoms.get_stress().tolist()
        except Exception:
            S = None

        # Extract ID
        if hasattr(data, "CSD_code"):
            csd_code = data.CSD_code
            if isinstance(csd_code, (list, tuple)):
                csd_code = csd_code[0]
            csd_code = str(csd_code)
        else:
            csd_code = None

        records.append(
            {
                "CSD_code": csd_code,
                "energy": float(E),
                "forces": F.tolist(),
                "stress": S,
            }
        )

    df = pd.DataFrame(records)
    if "CSD_code" in df.columns:
        df = df.set_index("CSD_code")

    return df

"""
# batched:
from torch_geometric.data import Batch

batched_data = Batch.from_data_list([data1, data2, data3])

pos = batched_data.pos.to(device)
z   = batched_data.z.to(device)
batch_idx = batched_data.batch.to(device)

# Direct forward pass (depends on predictor API)
out = predictor.model(pos=pos, z=z, batch=batch_idx)
"""

def run_all_tasks(predictor, dataloader, device="cuda"):
    """
    Given a UMA predictor and a PyG dataloader,
    evaluate ALL FAIRChem tasks and return a dict of DataFrames.
    """

    task_names = ["omol", "omat", "oc20", "omc", "odac"]

    results = {}

    for task in task_names:
        print(f"\n=== Running task: {task} ===")

        calc = FAIRChemCalculator(
            predictor,
            task_name=task
        )

        df = run_inference_for_task(calc, dataloader)
        results[task] = df

    return results


In [19]:
predictor = pretrained_mlip.get_predict_unit("uma-s-1p1", device="cuda")  
calc = FAIRChemCalculator(predictor, task_name="omol")

dataset = TMQMGStarDataset(
    root="data/datasets/TMQMG_Specto",
    block_3_only=True,
    prediction_type="vector",
    filter_type="all_visible_lambdas",
    prediction_params={"range": (380, 750)},
    vis_range=(380, 750)
)

num_samples = 50
indices = torch.randperm(len(dataset))[:num_samples]

subset_dataset = Subset(dataset, indices)

loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

results = run_all_tasks(predictor, loader)


=== Running task: omol ===



=== Running task: omat ===

=== Running task: oc20 ===

=== Running task: omc ===

=== Running task: odac ===


In [20]:
for task, df in results.items():
    print("Task:", task)
    print(df.head())
    print("="*75)

Task: omol
                 energy                                             forces  \
CSD_code                                                                     
PNOLTI    -53059.656938  [[-1.2442502975463867, 0.9363152384757996, -2....   
TIACPC    -50989.003018  [[0.6205921173095703, 0.8300570249557495, -0.7...   
KIBNES   -101435.946169  [[-0.03208615258336067, 0.003792482428252697, ...   
FUGNAW    -85182.955701  [[1.477144479751587, 0.3687206208705902, 0.823...   
HUBWEJ    -74350.909498  [[-0.0009387193131260574, 0.001375116524286568...   

                                                     stress  
CSD_code                                                     
PNOLTI    [-5.715177536010742, -6.1534247398376465, -6.9...  
TIACPC    [-4.592965602874756, -8.756617546081543, -7.79...  
KIBNES    [-8.669477462768555, -4.688338756561279, -10.5...  
FUGNAW    [-1.2452112436294556, -3.0697011947631836, -2....  
HUBWEJ    [-7.202732086181641, -5.079504013061523, -4.47...  
Task: om